# Newton Fundamentals — Build Once, Choose the Physics

In Article 2 you used Warp kernels and MuJoCo Warp to change **how many worlds you can advance together**. Newton asks a broader question: how do you describe a physical world once, give its different materials the solvers they need, and keep state, controls and observations understandable?

[Newton](https://github.com/newton-physics/newton/tree/v1.6.0) is an open-source physics engine built on NVIDIA Warp. It brings model authoring, rigid and deformable simulation, collision detection, world replication, sensors and inverse-kinematics tools into a Python workflow. MuJoCo Warp remains one important backend, not the entire engine.

You start here with a sphere and a ground plane, not a robot import. You will build a complete Newton simulation, inspect the actual arrays, change solver, and replicate the scene. In the next notebook you will apply these concepts to the **SO-101 and Seeed reBot DevArm**. The later clean-the-table exercise adds rigid/deformable coupling; it is a different task, not evidence supplied by this falling-sphere check.

## Outline

- Place Newton, Warp and MuJoCo Warp at the right layers.
- Build geometry with `ModelBuilder` and inspect a finalized `Model`.
- Allocate `State`, `Control` and `Contacts`; run an out-of-place substep loop.
- Run a headless drop with `SolverXPBD`, then reuse the model with `SolverMuJoCo`.
- Replicate a small world and understand where GPU scaling starts.
- Identify the boundaries of solver choice, sensors, differentiability and coupling.

## What you will learn

By the end of this notebook, you will be able to:

- Explain what Newton adds beyond calling `mjw.step` directly.
- Distinguish model topology, editable parameters, time-varying state and actuation.
- Choose a solver using its supported physics, not just a familiar class name.
- Advance two state buffers without reallocating or overwriting the input.
- Separate collision **detection**, contact **resolution** and measured task success.
- Read a CPU result without turning it into a claim about CUDA throughput.

## Setup: use the pinned Newton environment

Use the tutorial’s **Python 3.12 Newton environment**. The included MuJoCo baselines use the same pinned dependencies. From `tutorials/newton`, create and select the environment once:

```bash
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python --require-hashes -r notebooks/newton/requirements.lock.txt
# usd-core 26.3 on every platform: this starts Jupyter with the required Work limit.
PXR_WORK_THREAD_LIMIT=1 .venv/bin/jupyter lab --notebook-dir notebooks
```

Select `.venv/bin/python` as your notebook kernel in your editor. Start this notebook with its working directory set to **`newton/part3`**. The next cell checks that explicitly; it does not guess a repository path or change directories behind your back. Headless CPU execution is the default, so you do not need a display or a CUDA GPU to learn the API.

The later official-USD lessons require `PXR_WORK_THREAD_LIMIT=1` at process/kernel startup with `usd-core` 26.3 on every platform. Native parallel-parser aborts were observed on macOS ARM and Linux; the Windows guard is precautionary. This is a version-specific workaround, not an upstream fix or a physics tuning parameter. Restart a running Jupyter server/kernel after changing its launch environment; assigning the variable after importing `pxr` is not the verified recipe. This sphere lesson does not import a robot USD.

The executable examples target **Newton 1.6.0**, Warp 1.17.0 and MuJoCo/MuJoCo Warp 3.12.0. Source links below are pinned to that release rather than a moving `latest` page. `REFERENCE = True` is the read/run mode throughout these notebooks. This fundamentals notebook is complete in-line; later notebooks use the mode to choose finished `solutions/` entry points. After completing the external exercises, set it to `False` and restart/run their notebooks to verify **your** work.


In [ ]:
from pathlib import Path
import sys
from importlib.metadata import version

PART3 = Path.cwd().resolve()
assert PART3.name == "part3" and (PART3 / "final_check.py").is_file(), (
    "Start this kernel in <tutorial>/notebooks/newton/part3."
)
REFERENCE = True  # Read/run now; False selects your work in the exercise notebooks.

import numpy as np
import warp as wp
import newton
from newton.solvers import SolverBase, SolverMuJoCo, SolverXPBD

wp.init()
newton.use_coord_layout_targets = True  # Already the 1.6 default; make the policy explicit.
DEVICE = "cpu"  # Explicit and portable; this notebook does not benchmark CUDA.
print("Python:", sys.executable)
print("Working directory:", PART3)
for package in ("newton", "warp-lang", "mujoco", "mujoco-warp", "newton-usd-schemas", "usd-core"):
    print(f"{package}: {version(package)}")
assert version("newton") == "1.6.0", "Use newton/requirements.txt for the APIs below."
print("Devices:", [str(device) for device in wp.get_devices()])
print("Mode:", "reference read/run" if REFERENCE else "student review")

## Overview of the files

| File | How you use it |
|---|---|
| [Newton fundamentals](01__newton_fundamentals.ipynb) | This complete, runnable introduction; no TODO files needed |
| [MuJoCo to Newton](02__mujoco_to_newton.ipynb) | Next: complete the robot migration, Steps 0–9 |
| `newton_scene.py` | Your external scene-building exercises, Steps 0–5 |
| `so101_newton.py` | Your external simulation-loop exercises, Steps 6–9; both robot profiles |
| `solutions/newton_scene_solution.py`, `solutions/so101_newton_solution.py` | Finished references you can run directly; never copied over your exercises |
| `robots.py`, `pick_place_common.py`, `utils.py` | Robot profiles, reused controller/IK and asset helpers |
| [Final check](04__final_check.ipynb), `final_check.py` | Later: assert selected tasks and report GPU skips separately |
| `.generated/` | Generated assets and local executed notebook copies, not lesson source |

Keep your editor beside the notebooks when you reach the exercises. Here, run the cells in order and inspect each result before continuing.

## Step 0. Understand the stack you are choosing

```text
Your task: geometry, controller, observations, assertions
                         |
Newton: ModelBuilder -> Model + State + Control + Contacts
                         |
               solver / coupled solver
              /                       \
     SolverMuJoCo                   SolverVBD, XPBD, ...
       /       \
native MuJoCo  MuJoCo Warp
   (CPU)        (Warp; CUDA for throughput)
```

Warp supplies typed device arrays, kernels and execution machinery. MuJoCo Warp implements MuJoCo dynamics on that substrate. **Newton defines the surrounding physical model and solver interfaces**, so you can use more than one family of dynamics without inventing a second scene representation for every experiment.

| If you need… | What Newton gives you | What you must still verify |
|---|---|---|
| MJCF, URDF or USD assets | `add_mjcf`, `add_urdf`, `add_usd` on one builder | Importer coverage, units, joint frames, collision geometry and optional dependencies |
| Articulated rigid robots | Generalized-coordinate `SolverMuJoCo` or `SolverFeatherstone` | Actuators, contacts, constraints and solver-specific settings |
| Cloth, rods or soft objects | Solvers such as `SolverVBD` and dedicated material representations | Material parameters, stable timesteps and supported joints |
| Many independent environments | `ModelBuilder.replicate` and world-aware arrays | Contact/constraint capacity, memory, GPU workload and measured scaling |
| Shared observations and control | `newton.sensors`, `newton.ik`, Warp interoperability | Required fields and solver support for the quantities you request |
| Rigid/deformable interaction | Experimental coupled solvers with explicit ownership | Two-way feedback, contact coverage and task-level assertions |

If your workload only needs batched MuJoCo dynamics, direct MJWarp can still be the simpler choice. Newton becomes useful when the **model, materials and surrounding tools** need to grow together. Isaac Lab integration is a further layer for learning environments; you do not need Isaac Sim or Isaac Lab for these standalone examples.

See the release [overview](https://github.com/newton-physics/newton/blob/v1.6.0/docs/guide/overview.rst) and [solver capability matrix](https://github.com/newton-physics/newton/blob/v1.6.0/docs/solvers/index.rst).


## Step 1. Author a physical model

`ModelBuilder` is your authoring object. You add bodies, shapes, joints, particles and imported assets before choosing the final array layout and device.

A **body** has pose, velocity, mass and inertia. A **shape** supplies collision/visual geometry attached to a body. A shape with `body=-1` belongs to the world: that is how you make a static ground plane or table. Zero shape density alone is **not** a rule that turns an ordinary body static.

You will drop a sphere above a **Z-up** ground plane. The starting height and gravity must use the same axis. Here you set the total body mass and inertia explicitly, then use `density=0.0` to prevent the shape from adding more mass. In the robot exercise you will use this same distinction to preserve each cube's mass.

In [ ]:
RADIUS = 0.08
START_HEIGHT = 0.8
MASS = 0.2


def make_drop_builder():
    builder = newton.ModelBuilder(up_axis="Z", gravity=(0.0, 0.0, -9.81))
    builder.default_shape_cfg.gap = 0.0
    # Register before authoring: needed when this model later goes to MuJoCo.
    SolverMuJoCo.register_custom_attributes(builder)
    sphere = builder.add_body(
        xform=wp.transform((0.0, 0.0, START_HEIGHT), wp.quat_identity()),
        mass=MASS,
        inertia=wp.mat33(np.eye(3) * (2.0 / 5.0) * MASS * RADIUS**2),
        lock_inertia=True,
        label="falling_sphere",
    )
    builder.add_shape_sphere(
        body=sphere, radius=RADIUS,
        cfg=newton.ModelBuilder.ShapeConfig(density=0.0, mu=0.5, gap=0.0),
        label="sphere_collision",
    )
    return builder


builder = make_drop_builder()
builder.add_ground_plane()
print("Authored bodies:", builder.body_count, "shapes:", builder.shape_count)

## Step 2. Finalize and inspect the model

`finalize(device=...)` packs the authored topology and parameters into a `Model` backed by Warp arrays. You do not add a new body to an already allocated model; rebuild when topology changes.

**The model is not read-only.** Parameters such as gains or physical properties may be edited through supported APIs. A solver may cache derived data, so appropriate changes require `solver.notify_model_changed(...)` with the relevant `newton.ModelFlags`. Simply changing an array and assuming every backend rebuilt its caches is unsafe. Configure parameters before constructing your solver when possible.

`Model` is distinct from the instantaneous `State`: this separation lets you allocate fresh rollouts, reproduce initial conditions and batch work without re-importing every asset.

In [ ]:
model = builder.finalize(device=DEVICE)
print("Device:", model.device)
print("Bodies / shapes / worlds:", model.body_count, model.shape_count, model.world_count)
print("Body labels:", model.body_label)
print("Mass array:", model.body_mass.numpy())
print("Coordinate count:", model.joint_coord_count, "DOF count:", model.joint_dof_count)
assert model.body_count == 1
assert np.isclose(model.body_mass.numpy()[0], MASS)

## Step 3. Allocate State and Control

| Object | What you put there |
|---|---|
| `newton.Model` | Topology and model parameters, including initial joint configuration |
| `newton.State` | Time-varying body/joint/particle positions, velocities and force buffers |
| `newton.Control` | Position targets, velocity targets and applied joint forces |
| `newton.Contacts` | Collision geometry and optional contact data |
| `newton.solvers.SolverBase` | Common solver contract; concrete solvers implement dynamics |

Allocate two states for the out-of-place loop: the solver reads one and writes the other. Some backends permit aliasing, but two buffers make ownership clear and support the same pattern across this tutorial. `eval_fk` computes body transforms from joint coordinates; it does **not** advance time.

A rigid-body pose in `body_q` contains translation and an **xyzw** quaternion. A free joint therefore has a different coordinate count from its velocity DOFs. The later migration will map joint positions, DOFs and position-target offsets separately instead of slicing an assumed first six entries.

In [ ]:
state_0 = model.state()
state_1 = model.state()
control = model.control()
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)
initial_pose = state_0.body_q.numpy().copy()
print("body_q shape:", state_0.body_q.shape)
print("body_qd shape:", state_0.body_qd.shape)
print("Initial pose [x,y,z,qx,qy,qz,qw]:", initial_pose[0])
print("Position-target array:", control.joint_target_q.shape)
assert np.isclose(initial_pose[0, 2], START_HEIGHT)
assert state_0 is not state_1

## Step 4. Make contacts explicit

Collision **detection** finds potentially touching geometry. Contact **resolution** computes the resulting motion and forces as part of the dynamics solver. You can change the detector without pretending it is the integrator.

For XPBD you construct a `CollisionPipeline`, allocate its `Contacts`, and run `collide` on the **current input state before every physics substep**. Contact buffers have capacity limits; in a complex scene you must check capacity and overflow diagnostics, not assume missing contacts are harmless.

`SolverMuJoCo` also supports its own contact path. In Newton 1.6.0 the **native MuJoCo-C CPU stepping path does not consume Newton-generated contacts**. You will choose native detection explicitly for the CPU solver comparison below; using a Warp CPU device is not the same choice as setting `use_mujoco_cpu=True`. See the [release stepping paths](https://github.com/newton-physics/newton/blob/v1.6.0/newton/_src/solvers/mujoco/solver_mujoco.py#L4155-L4177).


In [ ]:
collision_pipeline = newton.CollisionPipeline(model)
contacts = collision_pipeline.contacts()
collision_pipeline.collide(state_0, contacts)
print("Initial rigid contacts:", contacts.rigid_contact_count.numpy())

## Step 5. Choose the solver by physics support

The shared `SolverBase.step` signature gives you a consistent loop, **not a promise that every solver can simulate every model**. Read the [v1.6.0 matrix](https://github.com/newton-physics/newton/blob/v1.6.0/docs/solvers/index.rst#L65-L155) before changing solver.

| Solver | Useful starting point | Boundary you should remember |
|---|---|---|
| `SolverMuJoCo` | Rigid bodies and generalized-coordinate articulations; MJCF workflows | No particles, cloth or soft bodies; **not differentiable** in this release |
| `SolverXPBD` | Position-based rigid/particle dynamics | Joint and deformable support differ from other solvers; **not differentiable** |
| `SolverFeatherstone` | Generalized-coordinate articulated dynamics | Basic differentiable workflows exist, with documented feature limits |
| `SolverSemiImplicit` | Rigid/particle/soft-body integration | Basic differentiable workflows; timestep and contact tuning still matter |
| `SolverVBD` | Cloth, rods and soft bodies; selected rigid/joint support | **Experimental**, limited joints, **not differentiable** |
| `SolverKamino` | Constrained rigid mechanisms | Experimental; check its dedicated guide |
| `SolverStyle3D` | Particle/cloth simulation | No rigid bodies, joints or differentiable simulation in this matrix |
| `SolverImplicitMPM` | Particle-based MPM material | No articulated rigid-body integration or differentiable simulation in this matrix |

Warp's automatic differentiation does not make every algorithm built on Warp differentiable. In particular, neither `SolverMuJoCo` nor `SolverVBD` provides that capability here. You start with XPBD because a primitive drop isolates the lifecycle without robot-import or actuator complexity.

In [ ]:
solver = SolverXPBD(model, iterations=8)
assert isinstance(solver, SolverBase)
print("Selected:", type(solver).__name__)

## Step 6. Run headless and assert the physical result

Use the same clock as the robot task: **50 Hz control, 10 physics substeps per frame**. Derive the substep duration; do not retain a different timestep from an imported MJCF.

The loop is `clear_forces → collide → step → swap`. Clearing force buffers prevents unintended accumulation; if you later add external forces, apply them **after clearing**. Swapping Python references makes the newest output the next input without copying the state arrays.

A successful process exit is not the test. You will require a finite state, normalized orientation, a sphere that actually fell, contacts that actually appeared, and a final center height close to the radius after settling. Sample contacts at every substep: a brief impact can disappear between controller-frame observations. The run allows time for transient motion to decay; do not loosen the final height assertion to accept a bouncing sphere.

In [ ]:
FPS = 50
SIM_SUBSTEPS = 10
FRAME_DT = 1.0 / FPS
SIM_DT = FRAME_DT / SIM_SUBSTEPS
NUM_FRAMES = 100
assert np.isclose(SIM_DT, 0.002)


def advance_frame(solver, state_0, state_1, control, contacts=None, pipeline=None, contact_history=None):
    for _ in range(SIM_SUBSTEPS):
        state_0.clear_forces()
        if pipeline is not None:
            pipeline.collide(state_0, contacts)
            if contact_history is not None:
                contact_history.append(int(contacts.rigid_contact_count.numpy()[0]))
        solver.step(state_0, state_1, control, contacts, SIM_DT)
        state_0, state_1 = state_1, state_0
    return state_0, state_1


contact_counts = []
for frame in range(NUM_FRAMES):
    state_0, state_1 = advance_frame(
        solver, state_0, state_1, control, contacts, collision_pipeline, contact_counts,
    )
wp.synchronize_device(model.device)
poses = state_0.body_q.numpy()
velocities = state_0.body_qd.numpy()
assert np.isfinite(poses).all() and np.isfinite(velocities).all()
assert np.allclose(np.linalg.norm(poses[:, 3:7], axis=1), 1.0, atol=1e-5)
assert poses[0, 2] < initial_pose[0, 2]
assert abs(float(poses[0, 2]) - RADIUS) < 0.02
assert max(contact_counts) > 0
assert np.linalg.norm(velocities[0]) < 0.05
print(f"XPBD: center_z={poses[0, 2]:.6f} m, simulated_time={NUM_FRAMES * FRAME_DT:.2f} s")
print("Measured maximum substep rigid contacts:", max(contact_counts))
print("PASS: finite, fallen and supported sphere on CPU")

## Step 7. Reuse the model with SolverMuJoCo

Now change the dynamics backend **without changing the geometry**. Allocate fresh state and controls from the same model so that the second run does not start from XPBD's final pose.

You request `use_mujoco_cpu=True` **and** `use_mujoco_contacts=True`. The native MuJoCo solver performs its own collision detection, so you pass `None` for Newton contacts and do not call the Newton pipeline in this run. For a CUDA MJWarp path you would use `use_mujoco_cpu=False`; native CPU versus MJWarp and Newton versus MuJoCo detection are separate decisions.

The simple drop should pass the same geometric test; it need not have an identical trajectory. Passing this primitive check does not validate your robot's frictional grasp. That stronger check belongs in the migration notebook.

In [ ]:
mujoco_solver = SolverMuJoCo(
    model, use_mujoco_cpu=True, use_mujoco_contacts=True,
    integrator="implicitfast", solver="newton",
)
mj_state_0, mj_state_1 = model.state(), model.state()
mj_control = model.control()
newton.eval_fk(model, model.joint_q, model.joint_qd, mj_state_0)
assert np.allclose(mj_state_0.body_q.numpy(), initial_pose)
for frame in range(NUM_FRAMES):
    mj_state_0, mj_state_1 = advance_frame(mujoco_solver, mj_state_0, mj_state_1, mj_control)
wp.synchronize_device(model.device)
mj_pose = mj_state_0.body_q.numpy()
assert np.isfinite(mj_pose).all() and np.isfinite(mj_state_0.body_qd.numpy()).all()
assert abs(float(mj_pose[0, 2]) - RADIUS) < 0.02
print(f"SolverMuJoCo / native CPU: center_z={mj_pose[0, 2]:.6f} m")
print("Measured difference from XPBD:", float(mj_pose[0, 2] - poses[0, 2]), "m")

## Step 8. Replicate worlds, not Python loops over simulations

A Newton model can contain multiple independent **worlds**. `replicate(template, world_count=...)` assigns each copy to its own world; you still step one solver over the packed model. World-aware storage and collision filtering keep independent copies separate even when they share coordinates.

Here you build **two CPU worlds** and inspect their world IDs. This is a correctness demonstration of replication, not a GPU scaling experiment. For visual separation, prefer viewer world offsets rather than moving physics far from the origin. Large batches also need measured capacity budgets and enough device memory.

`.numpy()` synchronizes/copies CUDA arrays back to the host. That is useful for these small assertions, but you should not put it in a device-resident policy loop and then call the result a throughput benchmark.

In [ ]:
template = make_drop_builder()
batch_builder = newton.ModelBuilder(up_axis="Z", gravity=(0.0, 0.0, -9.81))
SolverMuJoCo.register_custom_attributes(batch_builder)
batch_builder.replicate(template, world_count=2)
batch_builder.add_ground_plane()
batch_model = batch_builder.finalize(device=DEVICE)
batch_solver = SolverXPBD(batch_model, iterations=8)
batch_0, batch_1 = batch_model.state(), batch_model.state()
batch_control = batch_model.control()
batch_pipeline = newton.CollisionPipeline(batch_model)
batch_contacts = batch_pipeline.contacts()
newton.eval_fk(batch_model, batch_model.joint_q, batch_model.joint_qd, batch_0)
for frame in range(NUM_FRAMES):
    batch_0, batch_1 = advance_frame(batch_solver, batch_0, batch_1, batch_control, batch_contacts, batch_pipeline)
batch_pose = batch_0.body_q.numpy()
assert batch_model.world_count == 2 and batch_model.body_count == 2
assert np.isfinite(batch_pose).all() and np.isfinite(batch_0.body_qd.numpy()).all()
assert np.all(np.abs(batch_pose[:, 2] - RADIUS) < 0.02)
print("World IDs:", batch_model.body_world.numpy())
print("Final heights:", batch_pose[:, 2])
print("PASS: two independent CPU worlds; CUDA scaling remains unverified")

## Step 9. Extend the world deliberately

### Sensors and inverse kinematics

`newton.ik` solves kinematic objectives; it does not replace dynamics or guarantee a feasible contact-rich trajectory. In the next notebook you deliberately keep the Article 2 MuJoCo Jacobian-based IK helper so that you change one layer at a time.

Newton sensors offer transforms, IMU measurements, contact forces and camera observations. **Sensor support depends on the requested data and the solver.** For example, `SensorIMU` requests acceleration attributes; `SensorContact` needs contact-force support and `solver.update_contacts()` before the sensor update. Create sensors that request extended fields before allocating the relevant states/contacts. Do not infer that every backend populates every sensor merely because the class imports.

### Beyond a single solver

A robot, two cubes, a shirt and a cable do not all have the same numerical needs. In the advanced task, `SolverMuJoCo` owns the robot and cubes; `SolverVBD` owns cable bodies/`ROD` joints and cloth particles. Proxies expose both gripper jaws and the relevant cubes to the material solver and return contact reactions to the rigid solver. Each dynamic object retains one solver owner.

The migration notebook keeps the Article 2 MJCF robot as the migration control. The clean-table notebook instead imports the official SO-101 or reBot structured USD from pinned `newton-assets`, prepares the cable with `newton.Rod`, and opts into VBD compliant ALM. Its MJCF helper supplies controller IK and reBot robot-only gravity feedforward. See the [verification record](../VERIFICATION.md) for tested robot/device combinations.

Coupling must specify ownership, interface contact detection and feedback; calling two unrelated solvers is not a coupled simulation.

The release's experimental coupling API lives under `newton.solvers.experimental.coupled`. The advanced notebook makes that division concrete and checks measured grasp, lift, carry, release and settling for every object. You have **not** validated cloth, cable handling or two-way feedback by dropping a sphere.

See [Sensors](https://github.com/newton-physics/newton/blob/v1.6.0/docs/concepts/sensors.rst), [extended attributes](https://github.com/newton-physics/newton/blob/v1.6.0/docs/concepts/extended_attributes.rst), [coupling](https://github.com/newton-physics/newton/blob/v1.6.0/docs/concepts/coupling.rst), and the [1.6.0 migration notes](https://github.com/newton-physics/newton/blob/v1.6.0/CHANGELOG.md#L7-L97).


## Optional viewer: watch the same Newton state

Headless verification is complete without a window. To inspect the model interactively, set `RUN_VIEWER = True` on a local desktop with OpenGL support. The viewer reads Newton state; it is not `mujoco.viewer` and does not participate in the task assertions. If your notebook frontend cannot own the window, use the terminal command printed in the next notebook for the robot demo instead.

This opt-in cell uses the same initial state, XPBD solver and frame helper. It closes the window at the end rather than leaving an unbounded loop in Run All.

In [ ]:
RUN_VIEWER = False
if RUN_VIEWER:
    from newton.viewer import ViewerGL
    viewer = ViewerGL(width=960, height=640)
    view_0, view_1 = model.state(), model.state()
    newton.eval_fk(model, model.joint_q, model.joint_qd, view_0)
    view_solver = SolverXPBD(model, iterations=8)
    try:
        viewer.set_model(model)
        for frame in range(200):
            if not viewer.is_running():
                break
            view_0, view_1 = advance_frame(view_solver, view_0, view_1, control, contacts, collision_pipeline)
            viewer.begin_frame((frame + 1) * FRAME_DT)
            viewer.log_state(view_0)
            viewer.end_frame()
    finally:
        viewer.close()
else:
    print("Viewer not requested. Headless physics checks above are independent of rendering.")

## Troubleshooting

| Symptom | What you should check |
|---|---|
| `ModuleNotFoundError` or the wrong Newton version | Select the Newton kernel; `sys.executable` above identifies it. Do not rely on a shell's `python` alias. |
| The sphere starts on the ground | The model is Z-up: put height in Z, not Y. Check the initial pose before stepping. |
| No contacts or the sphere falls through | Did you add the ground, run detection before each XPBD substep, and pass the resulting contacts? |
| State never advances | Swap the two states; `eval_fk` alone computes poses, not dynamics. |
| Native MuJoCo does not respond to Newton contacts | Use MuJoCo detection on the native CPU path. Test external Newton contacts separately with MJWarp. |
| Changing a parameter has no effect | Check whether your solver needs `notify_model_changed` to rebuild derived data. |
| A solver accepts the model but your new material fails | Consult the capability matrix, timestep requirements and contact support before calling solvers interchangeable. |
| No CUDA device | The CPU lesson remains runnable; CUDA scaling, capture and performance remain unverified. |
| OpenGL/window errors | Leave the optional viewer disabled. Do not treat a rendering failure as a passed physics test or swallow assertion errors. |

## Recap

Before moving on, make sure you can answer:

- Which work belongs to the builder, model, state, control, collision pipeline and solver?
- Why does a zero-density shape differ from a world-attached static shape?
- Why do you reinitialize state when changing the solver?
- Which fields use coordinate offsets and which use velocity-DOF offsets?
- Why is a common solver API not a guarantee of identical physics, sensors or autodiff support?
- What did your two-world CPU result establish, and what did it **not** establish?

## Next

Continue to **[MuJoCo to Newton](02__mujoco_to_newton.ipynb)**. You will preserve the Article 2 task, import each robot through `ModelBuilder`, complete Steps 0–9 in the external Python files, and assert a real cube stack through `SolverMuJoCo`. Only after that do you add the advanced rigid/deformable task and run [Final check](04__final_check.ipynb).